Unifilar LV Diagram - Sendim (first version)
=
This notebook draws the low-voltage single-line diagram (unifilar BT) of UFV Sendim from Excel tables. It computes where every inverter block, inverter tag, AC cable tag, string tag and module power label goes, and writes AutoCAD scripts (`.scr`) that insert/write them. It doesn't read any DWG.

This is the first version of the Sendim diagram. `Unifilar LV optimized - Sendim.ipynb` is the later version (string distances, module power per string, AC cable Y fixed), and `Unifilar LV optimized 0D.py` is a script copy of this notebook (it also lists `INV-6.1.18` among the 17-string inverters).

**Workflow overview:**
1. Read the parameter, inverter, module power and string tables from the Sendim workbook.
2. Compute the inverter block, inverter tag and AC cable tag positions from the parameters.
3. Write the three inverter scripts.
4. Build the string table, apply the per-PST / per-inverter overrides of string count and pattern, and order the strings inside each inverter.
5. Put each string on a slot (row) of its inverter using `PATTERNS`, then write the string tag and module power scripts and an Excel copy of the table.

**Input prerequisites** (workbook `UFV Sendim - Unifilar BT.xlsx`, path in the first code cell):
- `Parametros` sheet: columns C:D, header on row 2. The values are read **by row position** from column D (`df_parameter.iloc[row, 1]`), so the row order must not change. What each row is used for (names inferred from the code):
  - 0 X origin, 1 Y origin, 2 X pitch per switchboard (PCS/QG pair), 3 X pitch between DJ columns, 4 Y pitch between DJs, 5 DJs per column
  - 6 / 7 inverter tag X / Y offset from the inverter
  - 8 not used
  - 9 / 10 string tag X / Y offset from the inverter, 11 string tag Y pitch per slot
  - 12 / 13 module power X / Y offset from the inverter, 14 module power Y pitch per slot
  - 15 AC cable tag X offset, 16 / 17 AC cable tag Y offset for QG 1 / QG 2, 18 AC cable tag Y step per DJ
- `INV` sheet: columns A:H, header on row 2. The **first column** is the inverter tag (`TAG`, e.g. `INV-1.1.03`), plus `UFV`, `PCS`, `QG`, `DJ`, `INV` and `Dist` (AC cable length, m).
- `Planilha1` sheet: `TAG INV` (e.g. `INV-1.1.01`) and `Module Power`. One row per inverter, otherwise the merge duplicates strings.
- `Sheet1` sheet: one row per string with `Tag completa` (e.g. `STR-1.1.01.01`), `UFV`, `PST`, `INV`, `STR`, `Type` (T / B / C) and `Type N` (1 / 2 / 3).
- String tags must be fixed-width `STR-U.P.II.SS` (1-digit UFV and PST, 2-digit INV, e.g. `STR-1.1.01.05`). The inverter tag is cut from characters 4-10 (`INV-1.1.01`) and the PST tag from characters 4-7 (`PST-1.1`). A 2-digit UFV or PST breaks both.

**Target drawing prerequisites** (to run the generated scripts, which use no DWG as input):
- A block named `Inversor` is defined in the drawing and is **not** set to *Scale uniformly*: the insert line answers X scale, Y scale and rotation (`1 1 0`). If the block has attributes, set `ATTREQ = 0`, because the script gives no attribute values.
- The layers `EMF_unifilar_inversor`, `EMF_TAG_inversor`, `CA_Cabos`, `TAG_strings` and `Potência_Módulo` exist. Each script starts with `-LAYER SET <layer>`, which fails on a missing layer.
- MText is written with the current text style (only height, justification, rotation and width are set).

Inputs and inverter positions
=
- `qty_strings = 18` and `pattern = 1` are the defaults for every string (18 strings per inverter, pattern 1). They are overridden per PST / inverter in the next cell.
- `inv_step = 9842.2311` is the vertical distance between two UFV blocks of the diagram. UFVs are stacked with `inv_step · (8 − UFV)`, so UFV 8 is at the bottom (offset 0) and UFV 1 at the top. The `8` is Sendim's UFV count.
- `PATTERNS`: the slot rows (1-based) used by the strings next to an inverter, for 18 (pattern 1), 17 (pattern 2) and 16 (pattern 3) strings. The gaps (4, 8-9, 13-14, 18, 22-23) separate the MPPT groups.
- Reads the four sheets (see the overview).
- Inverter block position:
  - `X = X0 + (2·PCS − 3 + QG) · pitch_switchboard + pitch_DJcolumn · trunc((DJ − 1) / DJs_per_column)`. Each PCS/QG pair gets its own column band (PCS 1/QG 1 → 0, PCS 1/QG 2 → 1, PCS 2/QG 1 → 2, ...), and the DJs wrap into a new column every `DJs_per_column`.
  - `Y = inv_step · (8 − UFV) + Y0 + ((DJ − 1) mod DJs_per_column) · pitch_DJ`. The expression `DJ − ceil(DJ/n)·n + n − 1` is just `(DJ − 1) mod n`.
- `X tag` / `Y tag`: inverter tag position = inverter position + fixed offset (rows 6 / 7).
- `X cabo`: AC cable tag column, one per PCS/QG pair (row 15 offset).
- The table is sorted by `UFV, PCS, QG, DJ` (the order of the script lines).
- `Y cabo` should be: DJ 1 of QG 1 / QG 2 → UFV offset + `Y0` + row 16 / 17, and each following DJ → previous row + row 18.
  - Warning: `Y cabo` starts as all-NaN and `np.where` evaluates `df_inv['Y cabo'].shift()` once, on that NaN column. Every row with `DJ ≠ 1` therefore ends up `NaN`, and those AC cable tags get `nan` coordinates in the script. The Sendim and Riodades notebooks replace this with a row loop.
- The saved output is a `PermissionError`: the workbook was open (locked) when the cell last ran.

In [1]:
import time

import pandas as pd

import math

import numpy as np

start = time.time()

qty_strings = 18
pattern = 1

inv_step = 9842.2311

PATTERNS = {
    1: [1,2,3,5,6,7,10,11,12,15,16,17,19,20,21,24,25,26],
    2: [1,2,3,5,6,7,10,11,12,15,16,17,19,20,21,24,25],
    3: [1,2,3,5,6,7,10,11,15,16,17,19,20,21,24,25]
}

with pd.ExcelFile(r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\302 - Zagope - UFV Sendim\01 - Projetos\01 - Em Confecção\02 - Edição\02 - Planilhas\UFV Sendim - Unifilar BT.xlsx") as xlsx:
    df_parameter = pd.read_excel(xlsx, sheet_name="Parametros", usecols="C:D", header=1)
    df_inv = pd.read_excel(xlsx, sheet_name="INV", usecols="A:H", header=1)
    df_pot = pd.read_excel(xlsx, sheet_name="Planilha1")
    df_type = pd.read_excel(xlsx, sheet_name="Sheet1")


df_inv['X'] = df_parameter.iloc[0, 1]+(df_inv["PCS"]*2-3+df_inv["QG"])*df_parameter.iloc[2, 1]+df_parameter.iloc[3, 1]*np.trunc((df_inv["DJ"]-1)/df_parameter.iloc[5, 1])
df_inv['Y'] = (inv_step * (8 - df_inv['UFV'])) + (df_parameter.iloc[1, 1] + (df_inv['DJ'] - np.ceil(df_inv['DJ'] / df_parameter.iloc[5, 1]) * df_parameter.iloc[5, 1] + df_parameter.iloc[5, 1] - 1.0) * df_parameter.iloc[4, 1])
df_inv['X tag'] = df_inv['X'] + df_parameter.iloc[6, 1]
df_inv['Y tag'] = df_inv['Y'] + df_parameter.iloc[7, 1]
df_inv['X cabo'] = df_parameter.iloc[0, 1]+(df_inv['PCS']*2+df_inv['QG']-3)*df_parameter.iloc[2, 1]+df_parameter.iloc[15, 1]
df_inv['Y cabo'] = np.nan

df_inv = df_inv.sort_values(by=["UFV", "PCS", "QG", "DJ"]).reset_index(drop=True)

df_inv['Y cabo'] = np.where(
    (df_inv['QG'] == 1) & (df_inv['DJ'] == 1),
    (inv_step * (8 - df_inv["UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[16, 1]),
    np.where(
        (df_inv['QG'] == 2) & (df_inv['DJ'] == 1),
        (inv_step * (8 - df_inv["UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[17, 1]),
        df_inv['Y cabo'].shift() + df_parameter.iloc[18,1]
    )
)

print(df_inv["Y cabo"])

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\Usuario\\Ecotechnee\\Office - Documentos\\Projetos\\302 - Zagope - UFV Sendim\\01 - Projetos\\01 - Em Confecção\\02 - Edição\\02 - Planilhas\\UFV Sendim - Unifilar BT.xlsx'

Inverter scripts, string placement and output files
=
**Inverter scripts**
- `TAG Cabo CA` = `CLV-UFV.PCS.INV.(L1-L2-L3) - <Dist>m` (INV zero-padded to 2 digits, length rounded to 0.1 m).
- `Script 1`: `-insert Inversor X,Y 1 1 0` (X scale 1, Y scale 1, rotation 0).
- `Script 2`: MText with the inverter tag (`TAG`), height 30, justification ML, rotation −90.
- `Script 3`: MText with the AC cable tag, height 20, justification TL, rotation 0.
  - Warning: `Script 1` joins X and Y with **no comma** (`X + Y`), so the insertion point is unreadable (e.g. `5000.073895.6177`). Fixed in the Sendim and Riodades notebooks.
- Writes `Script Inversor.scr` (layer `EMF_unifilar_inversor`), `Script TAG Inversor.scr` (layer `EMF_TAG_inversor`) and `Script Cabos CA.scr` (layer `CA_Cabos`) in `Desktop\Pessoal\Python\Diagrama Unifilar`.
- Renames the first `df_inv` column (the inverter tag) to `TAG INV` for the merge below.

**Strings**
- `df_strings` is built from `Sheet1`, with `Qty. of Strings = 18` and `PD = 1` for all. `PV`, `X`, `Y`, `XX`, `YY` and the script columns start empty.
- `TAG Especial` = `(Type) STR-...` (e.g. `(T) STR-1.1.01.01`). `TAG INV` / `TAG PST` are cut from the string tag.
- Left-merge with `Planilha1` on `TAG INV` adds `Module Power`.
- Hardcoded Sendim overrides (the inverter overrides run second, so they win):
  - PST-5.2, 7.2, 8.2 → 17 strings, pattern 2. PST-5.1, 7.1 → 16 strings, pattern 3.
  - INV-5.2.11, 5.2.15, 5.2.19 → 16 strings, pattern 3. INV-6.1.19, 6.1.20 → 17 strings, pattern 2. `INV-6.1.18` is missing here but present in the `.py` copy and in the Sendim notebook.
- T-C-B exception (from the code comment): in an inverter with **17 strings and exactly one C string**, the B strings get `Type N = 5` and the C string `Type N = 4`. Sorting by `Type N` then gives T-C-B (1-4-5) instead of the usual T-B-C (1-2-3), so the C string shares an MPPT with T strings.
  - The loop looks each row up by `TAG` and only updates the first match, so a duplicated string tag is only fixed once.
- Sorts the strings by `UFV, PST, INV, Type N` (no distance ordering in this version).
- `Position in group` = running count **per `PD`**, and `PV = PATTERNS[PD][position mod len(PATTERNS[PD])]` is the slot (row) of the string next to its inverter.
  - Warning: the counter runs per pattern over the whole plant, not per inverter. It only lines up if every inverter has exactly as many strings as its pattern has slots (18 / 17 / 16). One inverter with a different count shifts the slots of every following inverter with that pattern.
- Left-merge with the inverter `X`, `Y` on `TAG INV`. `df_strings` already has empty `X` / `Y` columns, so pandas names them `X_x` / `Y_x` (empty) and `X_y` / `Y_y` (inverter position).
  - `X_x` / `Y_x` are then overwritten with the **string tag** position: `X_inv + row 9`, `Y_inv + row 10 + row 11 · (PV − 1)`.
  - `XX` / `YY` hold the **module power** position: `X_inv + row 12`, `Y_inv + row 13 + row 14 · (PV − 1)`.
- `Script TAG String`: MText of `TAG Especial`, height 20, justification MR. `Script Potência`: MText of the module power, height 20, justification MC.
- Outputs in `Desktop\Pessoal\Python\Diagrama Unifilar`: `Diagrama Unifilar BT.xlsx` (the full string table), `tag string.scr` (layer `TAG_strings`) and `potencia.scr` (layer `Potência_Módulo`).
- Prints the time elapsed since the first cell.

In [ ]:

df_inv['TAG Cabo CA'] = "CLV-" + df_inv["UFV"].astype(str) + "." + \
    df_inv["PCS"].astype(str) + "." + \
    df_inv['INV'].astype(str).str.zfill(2) + \
    ".(L1-L2-L3) - " + \
    df_inv["Dist"].round(1).astype(str) + "m"

df_inv['Script 1'] = "-insert Inversor " + \
                     df_inv['X'].astype(str) + \
                     df_inv['Y'].astype(str) + \
                     " 1 1 0"

df_inv['Script 2'] = "(command-s \"._mtext\" \""+ df_inv['X tag'].astype(str) + "," + \
                     df_inv['Y tag'].astype(str) + \
                     "\" \"h\" 30  \"j\" \"ML\" \"r\" -90  \"w\" 0 \"" + \
                     df_inv['TAG'].astype(str) + \
                     "\" \"\")"

df_inv['Script 3'] = "(command-s \"._mtext\" \""+ df_inv['X cabo'].astype(str) + "," + \
                     df_inv['Y cabo'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"TL\" \"r\" 0  \"w\" 0 \"" + \
                     df_inv['TAG Cabo CA'].astype(str) + \
                     "\" \"\")"


with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Diagrama Unifilar\Script Inversor.scr", 'w') as f:
    f.write('-LAYER\nSET\nEMF_unifilar_inversor\n\n')
    f.write('\n'.join(df_inv["Script 1"].astype(str)) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Diagrama Unifilar\Script TAG Inversor.scr", 'w') as f:
    f.write('-LAYER\nSET\nEMF_TAG_inversor\n\n')
    f.write('\n'.join(df_inv["Script 2"].astype(str)) + '\n')

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Diagrama Unifilar\Script Cabos CA.scr", 'w') as f:
    f.write('-LAYER\nSET\nCA_Cabos\n\n')
    f.write('\n'.join(df_inv["Script 3"].astype(str)) + '\n')

df_inv = df_inv.rename(columns={df_inv.columns[0]: "TAG INV"})



 # Strings TAGs
df_strings = pd.DataFrame({
    'TAG': df_type["Tag completa"],
    'TAG Especial': None,
    'TAG INV': None,
    'TAG PST': None,
    'UFV': df_type["UFV"],
    'PST': df_type["PST"],
    'INV': df_type["INV"],
    'STR': df_type["STR"],
    'Type': df_type["Type"],
    'Type N': df_type["Type N"],
    'Qty. of Strings': qty_strings,
    'PD': pattern,
    'PV': None,
    'X': None,
    'Y': None,
    'XX': None,
    'YY': None,
    'Script TAG String': None,
    'Script Potência': None
})

df_strings["TAG Especial"] = "("+df_strings['Type'].astype(str)+") "+df_strings['TAG'].astype(str)
df_strings["TAG INV"] = "INV-" + df_strings["TAG"].astype(str).str.slice(4, 10)
df_strings["TAG PST"] = "PST-" + df_strings["TAG"].astype(str).str.slice(4, 7)
df_strings = df_strings.merge(df_pot, on="TAG INV", how="left")

df_strings["Qty. of Strings"] = np.where(
    (df_strings["TAG PST"] == "PST-5.2") | (df_strings["TAG PST"] == "PST-7.2") | (df_strings["TAG PST"] == "PST-8.2"),
    17,
    np.where(
        (df_strings["TAG PST"] == "PST-5.1") | (df_strings["TAG PST"] == "PST-7.1"),
        16,
        df_strings["Qty. of Strings"]
    )
)

df_strings["PD"] = np.where(
    (df_strings["TAG PST"] == "PST-5.2") | (df_strings["TAG PST"] == "PST-7.2") | (df_strings["TAG PST"] == "PST-8.2"),
    2,
    np.where(
        (df_strings["TAG PST"] == "PST-5.1") | (df_strings["TAG PST"] == "PST-7.1"),
        3,
        df_strings["PD"]
    )
)

df_strings["Qty. of Strings"] = np.where(
    (df_strings["TAG INV"] == "INV-5.2.11") | (df_strings["TAG INV"] == "INV-5.2.15") | (df_strings["TAG INV"] == "INV-5.2.19"),
    16,
    np.where(
        (df_strings["TAG INV"] == "INV-6.1.19") | (df_strings["TAG INV"] == "INV-6.1.20"),
        17,
        df_strings["Qty. of Strings"]
    )
)

df_strings["PD"] = np.where(
    (df_strings["TAG INV"] == "INV-5.2.11") | (df_strings["TAG INV"] == "INV-5.2.15") | (df_strings["TAG INV"] == "INV-5.2.19"),
    3,
    np.where(
        (df_strings["TAG INV"] == "INV-6.1.19") | (df_strings["TAG INV"] == "INV-6.1.20"),
        2,
        df_strings["PD"]
    )
)

all_tags = df_strings["TAG INV"].unique()
c_counts = (df_strings[df_strings["Type N"] == 3]["TAG INV"].value_counts().reindex(all_tags, fill_value=0))

for tag in df_strings["TAG"]:
    index = df_strings[df_strings["TAG"] == tag].index[0]
    tag_inv = df_strings.at[index, "TAG INV"]
    c_count = c_counts[tag_inv]
    if df_strings.at[index, "Qty. of Strings"] == 17 and c_count == 1:
        if df_strings.at[index, "Type N"] == 2:
            df_strings.at[index, "Type N"] = 5
        elif df_strings.at[index, "Type N"] == 3:
            df_strings.at[index, "Type N"] = 4


df_strings = df_strings.sort_values(by=["UFV", "PST", "INV", "Type N"]).reset_index(drop=True)

df_strings["Position in group"] = df_strings.groupby("PD").cumcount()

df_strings['PV'] = df_strings.apply(
    lambda row: PATTERNS[row['PD']][row['Position in group'] % len(PATTERNS[row['PD']])],
    axis=1
)

df_strings = df_strings.merge(df_inv[["TAG INV", "X", "Y"]], on="TAG INV", how="left")

df_strings['X_x'] = df_strings['X_y'].astype(float)+df_parameter.iloc[9,1]
df_strings['Y_x'] = df_strings['Y_y'].astype(float)+df_parameter.iloc[10,1]+df_parameter.iloc[11, 1]*(df_strings["PV"].astype(float)-1)
df_strings['XX'] = df_strings['X_y'].astype(float)+df_parameter.iloc[12,1]
df_strings['YY'] = df_strings['Y_y'].astype(float)+df_parameter.iloc[13,1]+df_parameter.iloc[14, 1]*(df_strings["PV"].astype(float)-1)

df_strings['Script TAG String'] = "(command-s \"._mtext\" \""+ df_strings['X_x'].astype(str) + "," + \
                     df_strings['Y_x'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"MR\" \"r\" 0  \"w\" 0 \"" + \
                     df_strings['TAG Especial'].astype(str) + \
                     "\" \"\")"

df_strings['Script Potência'] = "(command-s \"._mtext\" \""+ df_strings['XX'].astype(str) + "," + \
                     df_strings['YY'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     df_strings['Module Power'].astype(str) + \
                     "\" \"\")"

df_strings.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Diagrama Unifilar\Diagrama Unifilar BT.xlsx")

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Diagrama Unifilar\tag string.scr", 'w') as f:
    f.write('-LAYER\nSET\nTAG_strings\n\n')
    f.write('\n'.join(df_strings["Script TAG String"].astype(str)) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Diagrama Unifilar\potencia.scr", 'w') as f:
    f.write('-LAYER\nSET\nPotência_Módulo\n\n')
    f.write('\n'.join(df_strings["Script Potência"].astype(str)) + '\n')


end = time.time()
elapsed = end - start
print(elapsed)